# CareTrack AI — Machine Learning Pipeline

This notebook demonstrates the complete Machine Learning pipeline for the **CareTrack AI** Platform. The objective is to evaluate patient chronic care progress and categorize adherence archetypes using daily log telemetry.

### Models Implemented:
1. **Random Forest Classifier**: Categorizes patient progress (Improving, Stable, Deteriorating) based on multi-dimensional telemetry (medication compliance, exercise consistency, diet compliance, biometric fluctuations).
2. **Logistic Regression Classifier**: Supervised classification baseline.
3. **K-Means Clustering**: Clusters patients into care archetypes for clinician overview.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, accuracy_score, silhouette_score, confusion_matrix
import joblib

## 1. Synthetic Dataset Generation

In [ ]:
def generate_dataset(num_samples=1000, seed=42):
    np.random.seed(seed)
    med_adherence = np.random.uniform(0.1, 1.0, num_samples)
    exercise_consistency = np.random.uniform(0.0, 1.0, num_samples)
    diet_compliance = np.random.uniform(0.1, 1.0, num_samples)
    sleep_avg = np.random.normal(7.0, 1.0, num_samples)
    sleep_avg = np.clip(sleep_avg, 4.0, 10.0)
    
    # Correlate changes with compliance
    weight_change = -3.0 * med_adherence - 2.0 * exercise_consistency + np.random.normal(1.5, 0.8, num_samples)
    glucose_change = -50.0 * med_adherence - 30.0 * diet_compliance + np.random.normal(30, 15, num_samples)
    
    # Define rules + noise to get target labels (0=Deteriorating, 1=Stable, 2=Improving)
    score = (2.5 * med_adherence + 1.5 * exercise_consistency + 1.5 * diet_compliance - 
             0.5 * weight_change - 0.02 * glucose_change + np.random.normal(0, 0.5, num_samples))
    
    labels = []
    for s in score:
        if s > 3.8:
            labels.append(2)  # Improving
        elif s < 2.2:
            labels.append(0)  # Deteriorating
        else:
            labels.append(1)  # Stable
            
    return pd.DataFrame({
        'med_adherence': med_adherence,
        'exercise_consistency': exercise_consistency,
        'diet_compliance': diet_compliance,
        'weight_change': weight_change,
        'glucose_change': glucose_change,
        'sleep_avg': sleep_avg,
        'progress_label': labels
    })

df = generate_dataset(1200)
df.head()

## 2. Preprocessing & Splits

In [ ]:
X = df.drop(columns=['progress_label'])
y = df['progress_label']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Train size: {X_train.shape[0]}, Test size: {X_test.shape[0]}")

## 3. Supervised Classification (Random Forest vs Logistic Regression)

In [ ]:
print("--- Random Forest ---")
rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train_scaled, y_train)
rf_preds = rf.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, rf_preds) * 100:.2f}%")
print(classification_report(y_test, rf_preds, target_names=['Deteriorating', 'Stable', 'Improving']))

print("--- Logistic Regression ---")
lr = LogisticRegression(max_iter=500, random_state=42)
lr.fit(X_train_scaled, y_train)
lr_preds = lr.predict(X_test_scaled)
print(f"Accuracy: {accuracy_score(y_test, lr_preds) * 100:.2f}%")

## 4. Feature Importances

In [ ]:
importances = rf.feature_importances_
feat_importances = pd.Series(importances, index=X.columns)
feat_importances.sort_values().plot(kind='barh', color='teal')
plt.title('Random Forest Feature Importance')
plt.xlabel('Importance Value')
plt.show()

## 5. Unsupervised Clustering (K-Means)

In [ ]:
compliance_df = df[['med_adherence', 'exercise_consistency', 'diet_compliance']]
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
kmeans.fit(compliance_df)

sil = silhouette_score(compliance_df, kmeans.labels_)
print(f"K-Means Silhouette Score: {sil:.3f}")

# Show mean compliance rates per group
compliance_df['group'] = kmeans.labels_
print(compliance_df.groupby('group').mean())

## 6. Saving Models

In [ ]:
save_dir = '../app/ml/models'
os.makedirs(save_dir, exist_ok=True)

joblib.dump(rf, os.path.join(save_dir, 'rf_classifier.joblib'))
joblib.dump(scaler, os.path.join(save_dir, 'scaler.joblib'))
joblib.dump(kmeans, os.path.join(save_dir, 'kmeans_cluster.joblib'))
print("Pipeline serialisation complete!")